# NanoForge on a Colab L4 GPU

Choose **Runtime → Change runtime type → L4 GPU** first, then run the cell below. It preserves Colab's CUDA PyTorch, checks the actual GPU, runs the real CUDA/FP16 checkpoint smoke tests, imports Dolly, prepares uint16 tokens and trains from random weights.

The L4 profile uses Mini, Magic 8k, context 512, batch 32 and accumulation 2 (effective batch 64). FP16 autocast and gradient scaling accelerate training; TF32 is enabled for remaining FP32 matmuls on supported GPUs. Parameters and exported checkpoint tensors remain FP32. PyTorch selects a supported SDPA attention kernel; CUDA kernels are not forced.

Set `CUSTOM_MODEL` to edit architecture below one billion parameters, reducing batch size as needed. The parameter limit does not guarantee a model fits GPU memory. Set `RESUME` to a full NanoForge Magic 8k `.pt` checkpoint to continue it; saved architecture/tokenizer are retained, and steps are additional updates. Resume data must be available locally.

The setup imports **Dolly only**, not the entire large dataset catalog. `/content` is temporary: download outputs before disconnecting, or change `RUN_HOME` to an already mounted Google Drive directory. Drive cache reads can be slower. Actual L4 execution is checked by the runtime smoke tests; the development environment has no GPU.

In [ ]:
from pathlib import Path
import subprocess

# EDIT THESE SETTINGS:
TRAINING_STEPS = 1000
BATCH_SIZE = 32
GRADIENT_ACCUMULATION = 2
CUSTOM_MODEL = None
# CUSTOM_MODEL = dict(n_layer=12, n_head=8, n_embd=768, block_size=512, dropout=0.1)
# For larger models, start with BATCH_SIZE = 1 and increase accumulation.
RESUME = None  # Example: '/content/drive/MyDrive/model.pt'
RUN_HOME = '/content/nanoforge-l4-run'

REPO = Path('/content/nanoforge')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/bananaosint/nanoforge.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)

# Load the committed helper into this notebook kernel; training uses subprocesses.
startup = {'__name__': 'nanoforge_colab_l4'}
script = REPO/'scripts/colab_l4.py'
exec(compile(script.read_text(),str(script),'exec'),startup)
result = startup['run_l4'](
    repo=REPO, home=RUN_HOME, steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE, accumulation=GRADIENT_ACCUMULATION,
    custom_model=CUSTOM_MODEL, resume=RESUME,
)

from google.colab import files
from IPython.display import FileLink, display
print('Download the .pt, then select Import .pt checkpoint in Studio.')
files.download(result['checkpoint'])
display(FileLink(result['checkpoint']))
display(FileLink(result['metadata']))

## Open in the local Studio

Use **Train & Monitor → Import .pt checkpoint**. It validates the weights and reconstructs the listing metadata. Alternatively copy both the `.pt` and `.json` into the local `checkpoints/` directory with Studio stopped. CPU inference uses FP32; the file does not require CUDA or XLA to deserialize. Choose local datasets before continuing training.

Each interval saves another checkpoint; check available disk space for large models. No ZIP is made automatically, avoiding a second large copy. If an automatic browser download is blocked or too large, use Colab's Files panel or the links and preserve the output before ending the runtime.